# Training Suit TinyML

Notebook ini menjalankan pipeline resmi dari `training/train.py`: membuat dataset sintetis, melatih model Keras, mengonversi ke TFLite, memverifikasi hasil, dan membuat ZIP artefak.

> Jalankan semua sel secara berurutan. Saat diminta, unggah file `training/train.py` dari proyek. Model ini mempelajari encoding tombol untuk tujuan pembelajaran deployment TinyML; bukan pengenalan tangan atau kamera.

## 1. Periksa lingkungan

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("tensorflow") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow", "numpy"], check=True)

import numpy as np
import tensorflow as tf

print("Python    :", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("NumPy     :", np.__version__)

## 2. Siapkan script training

Jika `train.py` belum ada di sesi Colab, pemilih file akan muncul. Unggah `training/train.py` dari folder proyek ini.

In [ ]:
from pathlib import Path

script = next((path for path in [Path("train.py"), Path("training/train.py")] if path.is_file()), Path("train.py"))
if not script.is_file():
    try:
        colab_files = importlib.import_module("google.colab").files
    except ModuleNotFoundError as exc:
        raise FileNotFoundError("Jalankan Jupyter dari root proyek atau salin training/train.py ke folder notebook.") from exc
    uploaded = colab_files.upload()
    if "train.py" not in uploaded:
        raise FileNotFoundError("File yang dipilih harus bernama train.py")

print("Script siap:", script.resolve())

## 3. Training dan ekspor

Hasil ditulis ke folder `suit_output/`. Ubah nilai `epochs` hanya jika ingin melakukan eksperimen.

In [ ]:
epochs = 180
subprocess.run([sys.executable, str(script), "--out", "suit_output", "--epochs", str(epochs)], check=True)

## 4. Verifikasi artefak

Sel ini memastikan file penting tersedia, model memenuhi batas kualitas minimum, dan header C++ identik dengan model TFLite.

In [ ]:
import json
import re

output = Path("suit_output")
required = [
    output / "data/dataset_sintetis.csv",
    output / "models/suit.keras",
    output / "models/suit_float32.tflite",
    output / "wokwi/model_data.h",
    output / "reports/training_metrics.json",
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, f"Artefak tidak lengkap: {missing}"

metrics = json.loads((output / "reports/training_metrics.json").read_text())
binary = (output / "models/suit_float32.tflite").read_bytes()
header = (output / "wokwi/model_data.h").read_text()
header_bytes = bytes(int(value, 16) for value in re.findall(r"0x([0-9a-f]{2})", header))

assert metrics["test_accuracy"] >= 0.90
assert metrics["max_abs_keras_tflite_error"] < 1e-4
assert set(metrics["ops"]) <= {"FULLY_CONNECTED", "SOFTMAX"}
assert header_bytes == binary

print(f"Akurasi uji : {metrics['test_accuracy']:.2%}")
print(f"Ukuran model: {metrics['tflite_bytes']} bytes")
print("Operator     :", ", ".join(metrics["ops"]))
print("Verifikasi  : LULUS")

## 5. Unduh hasil

ZIP berisi dataset, model Keras, TFLite, header C++, dan laporan evaluasi. Untuk memperbarui firmware, gunakan `suit_output/wokwi/model_data.h`.

In [ ]:
import shutil

archive = shutil.make_archive("hasil_training_suit", "zip", output)
print("Arsip siap:", archive)

try:
    colab_files = importlib.import_module("google.colab").files
    colab_files.download(archive)
except ModuleNotFoundError:
    print("Notebook lokal: ambil file ZIP dari path di atas.")